# 05 — Train Conditional Gan / 生成对抗网络

**Chapter 17 — File 5 of 6 / 第17章 — 第5个文件（共6个）**

---

## Summary / 总结

This script demonstrates **example of training an conditional gan on the fashion mnist dataset**.

本脚本演示 **example of training an conditional gan on the fashion mnist dataset**。

---
## Step 1 — example of training an conditional gan on the fashion mnist dataset

In [ ]:
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.datasets.fashion_mnist import load_data
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Dropout
from keras.layers import Embedding
from keras.layers import Concatenate

---
## Step 2 — define the standalone discriminator model

In [ ]:
def define_discriminator(in_shape=(28,28,1), n_classes=10):

---
## Step 3 — label input

In [ ]:
in_label = Input(shape=(1,))

---
## Step 4 — embedding for categorical input

In [ ]:
li = Embedding(n_classes, 50)(in_label)

---
## Step 5 — scale up to image dimensions with linear activation

In [ ]:
n_nodes = in_shape[0] * in_shape[1]
	li = Dense(n_nodes)(li)

---
## Step 6 — reshape to additional channel

In [ ]:
li = Reshape((in_shape[0], in_shape[1], 1))(li)

---
## Step 7 — image input

In [ ]:
in_image = Input(shape=in_shape)

---
## Step 8 — concat label as a channel

In [ ]:
merge = Concatenate()([in_image, li])

---
## Step 9 — downsample

In [ ]:
fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(merge)
	fe = LeakyReLU(alpha=0.2)(fe)

---
## Step 10 — downsample

In [ ]:
fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(fe)
	fe = LeakyReLU(alpha=0.2)(fe)

---
## Step 11 — flatten feature maps

In [ ]:
fe = Flatten()(fe)

---
## Step 12 — dropout

In [ ]:
fe = Dropout(0.4)(fe)

---
## Step 13 — output

In [ ]:
out_layer = Dense(1, activation='sigmoid')(fe)

---
## Step 14 — define model

In [ ]:
model = Model([in_image, in_label], out_layer)

---
## Step 15 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

---
## Step 16 — define the standalone generator model

In [ ]:
def define_generator(latent_dim, n_classes=10):

---
## Step 17 — label input

In [ ]:
in_label = Input(shape=(1,))

---
## Step 18 — embedding for categorical input

In [ ]:
li = Embedding(n_classes, 50)(in_label)

---
## Step 19 — linear multiplication

In [ ]:
n_nodes = 7 * 7
	li = Dense(n_nodes)(li)

---
## Step 20 — reshape to additional channel

In [ ]:
li = Reshape((7, 7, 1))(li)

---
## Step 21 — image generator input

In [ ]:
in_lat = Input(shape=(latent_dim,))

---
## Step 22 — foundation for 7x7 image

In [ ]:
n_nodes = 128 * 7 * 7
	gen = Dense(n_nodes)(in_lat)
	gen = LeakyReLU(alpha=0.2)(gen)
	gen = Reshape((7, 7, 128))(gen)

---
## Step 23 — merge image gen and label input

In [ ]:
merge = Concatenate()([gen, li])

---
## Step 24 — upsample to 14x14

In [ ]:
gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(merge)
	gen = LeakyReLU(alpha=0.2)(gen)

---
## Step 25 — upsample to 28x28

In [ ]:
gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(gen)
	gen = LeakyReLU(alpha=0.2)(gen)

---
## Step 26 — output

In [ ]:
out_layer = Conv2D(1, (7,7), activation='tanh', padding='same')(gen)

---
## Step 27 — define model

In [ ]:
model = Model([in_lat, in_label], out_layer)
	return model

---
## Step 28 — define the combined generator and discriminator model, for updating the generator

In [ ]:
def define_gan(g_model, d_model):

---
## Step 29 — make weights in the discriminator not trainable

In [ ]:
d_model.trainable = False

---
## Step 30 — get noise and label inputs from generator model

In [ ]:
gen_noise, gen_label = g_model.input

---
## Step 31 — get image output from the generator model

In [ ]:
gen_output = g_model.output

---
## Step 32 — connect image output and label input from generator as inputs to discriminator

In [ ]:
gan_output = d_model([gen_output, gen_label])

---
## Step 33 — define gan model as taking noise and label and outputting a classification

In [ ]:
model = Model([gen_noise, gen_label], gan_output)

---
## Step 34 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

---
## Step 35 — load fashion mnist images

In [ ]:
def load_real_samples():

---
## Step 36 — load dataset

In [ ]:
(trainX, trainy), (_, _) = load_data()

---
## Step 37 — expand to 3d, e.g. add channels

In [ ]:
X = expand_dims(trainX, axis=-1)

---
## Step 38 — convert from ints to floats

In [ ]:
X = X.astype('float32')

---
## Step 39 — scale from [0,255] to [-1,1]

In [ ]:
X = (X - 127.5) / 127.5
	return [X, trainy]

---
## Step 40 — # select real samples

In [ ]:
def generate_real_samples(dataset, n_samples):

---
## Step 41 — split into images and labels

In [ ]:
images, labels = dataset

---
## Step 42 — choose random instances

In [ ]:
ix = randint(0, images.shape[0], n_samples)

---
## Step 43 — select images and labels

In [ ]:
X, labels = images[ix], labels[ix]

---
## Step 44 — generate class labels

In [ ]:
y = ones((n_samples, 1))
	return [X, labels], y

---
## Step 45 — generate points in latent space as input for the generator

In [ ]:
def generate_latent_points(latent_dim, n_samples, n_classes=10):

---
## Step 46 — generate points in the latent space

In [ ]:
x_input = randn(latent_dim * n_samples)

---
## Step 47 — reshape into a batch of inputs for the network

In [ ]:
z_input = x_input.reshape(n_samples, latent_dim)

---
## Step 48 — generate labels

In [ ]:
labels = randint(0, n_classes, n_samples)
	return [z_input, labels]

---
## Step 49 — use the generator to generate n fake examples, with class labels

In [ ]:
def generate_fake_samples(generator, latent_dim, n_samples):

---
## Step 50 — generate points in latent space

In [ ]:
z_input, labels_input = generate_latent_points(latent_dim, n_samples)

---
## Step 51 — predict outputs

In [ ]:
images = generator.predict([z_input, labels_input])

---
## Step 52 — create class labels

In [ ]:
y = zeros((n_samples, 1))
	return [images, labels_input], y

---
## Step 53 — train the generator and discriminator

In [ ]:
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=100, n_batch=128):
	bat_per_epo = int(dataset[0].shape[0] / n_batch)
	half_batch = int(n_batch / 2)

---
## Step 54 — manually enumerate epochs

In [ ]:
for i in range(n_epochs):

---
## Step 55 — enumerate batches over the training set

In [ ]:
for j in range(bat_per_epo):

---
## Step 56 — get randomly selected 'real' samples

In [ ]:
[X_real, labels_real], y_real = generate_real_samples(dataset, half_batch)

---
## Step 57 — update discriminator model weights

In [ ]:
d_loss1, _ = d_model.train_on_batch([X_real, labels_real], y_real)

---
## Step 58 — generate 'fake' examples

In [ ]:
[X_fake, labels], y_fake = generate_fake_samples(g_model, latent_dim, half_batch)

---
## Step 59 — update discriminator model weights

In [ ]:
d_loss2, _ = d_model.train_on_batch([X_fake, labels], y_fake)

---
## Step 60 — prepare points in latent space as input for the generator

In [ ]:
[z_input, labels_input] = generate_latent_points(latent_dim, n_batch)

---
## Step 61 — create inverted labels for the fake samples

In [ ]:
y_gan = ones((n_batch, 1))

---
## Step 62 — update the generator via the discriminator's error

In [ ]:
g_loss = gan_model.train_on_batch([z_input, labels_input], y_gan)

---
## Step 63 — summarize loss on this batch

In [ ]:
print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' %
				(i+1, j+1, bat_per_epo, d_loss1, d_loss2, g_loss))

---
## Step 64 — save the generator model

In [ ]:
g_model.save('cgan_generator.h5')

---
## Step 65 — size of the latent space

In [ ]:
latent_dim = 100

---
## Step 66 — create the discriminator

In [ ]:
d_model = define_discriminator()

---
## Step 67 — create the generator

In [ ]:
g_model = define_generator(latent_dim)

---
## Step 68 — create the gan

In [ ]:
gan_model = define_gan(g_model, d_model)

---
## Step 69 — load image data

In [ ]:
dataset = load_real_samples()

---
## Step 70 — train model

In [ ]:
train(g_model, d_model, gan_model, dataset, latent_dim)

---
## Learning Notes / 学习笔记

- **概念**: example of training an conditional gan on the fashion mnist dataset 是机器学习中的常用技术。  
  *example of training an conditional gan on the fashion mnist dataset is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Conditional Gan / 生成对抗网络
# Complete Code / 完整代码
# ===============================

# example of training an conditional gan on the fashion mnist dataset
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.datasets.fashion_mnist import load_data
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Dropout
from keras.layers import Embedding
from keras.layers import Concatenate

# define the standalone discriminator model
def define_discriminator(in_shape=(28,28,1), n_classes=10):
	# label input
	in_label = Input(shape=(1,))
	# embedding for categorical input
	li = Embedding(n_classes, 50)(in_label)
	# scale up to image dimensions with linear activation
	n_nodes = in_shape[0] * in_shape[1]
	li = Dense(n_nodes)(li)
	# reshape to additional channel
	li = Reshape((in_shape[0], in_shape[1], 1))(li)
	# image input
	in_image = Input(shape=in_shape)
	# concat label as a channel
	merge = Concatenate()([in_image, li])
	# downsample
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(merge)
	fe = LeakyReLU(alpha=0.2)(fe)
	# downsample
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	# flatten feature maps
	fe = Flatten()(fe)
	# dropout
	fe = Dropout(0.4)(fe)
	# output
	out_layer = Dense(1, activation='sigmoid')(fe)
	# define model
	model = Model([in_image, in_label], out_layer)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

# define the standalone generator model
def define_generator(latent_dim, n_classes=10):
	# label input
	in_label = Input(shape=(1,))
	# embedding for categorical input
	li = Embedding(n_classes, 50)(in_label)
	# linear multiplication
	n_nodes = 7 * 7
	li = Dense(n_nodes)(li)
	# reshape to additional channel
	li = Reshape((7, 7, 1))(li)
	# image generator input
	in_lat = Input(shape=(latent_dim,))
	# foundation for 7x7 image
	n_nodes = 128 * 7 * 7
	gen = Dense(n_nodes)(in_lat)
	gen = LeakyReLU(alpha=0.2)(gen)
	gen = Reshape((7, 7, 128))(gen)
	# merge image gen and label input
	merge = Concatenate()([gen, li])
	# upsample to 14x14
	gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(merge)
	gen = LeakyReLU(alpha=0.2)(gen)
	# upsample to 28x28
	gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(gen)
	gen = LeakyReLU(alpha=0.2)(gen)
	# output
	out_layer = Conv2D(1, (7,7), activation='tanh', padding='same')(gen)
	# define model
	model = Model([in_lat, in_label], out_layer)
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model):
	# make weights in the discriminator not trainable
	d_model.trainable = False
	# get noise and label inputs from generator model
	gen_noise, gen_label = g_model.input
	# get image output from the generator model
	gen_output = g_model.output
	# connect image output and label input from generator as inputs to discriminator
	gan_output = d_model([gen_output, gen_label])
	# define gan model as taking noise and label and outputting a classification
	model = Model([gen_noise, gen_label], gan_output)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

# load fashion mnist images
def load_real_samples():
	# load dataset
	(trainX, trainy), (_, _) = load_data()
	# expand to 3d, e.g. add channels
	X = expand_dims(trainX, axis=-1)
	# convert from ints to floats
	X = X.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	return [X, trainy]

# # select real samples
def generate_real_samples(dataset, n_samples):
	# split into images and labels
	images, labels = dataset
	# choose random instances
	ix = randint(0, images.shape[0], n_samples)
	# select images and labels
	X, labels = images[ix], labels[ix]
	# generate class labels
	y = ones((n_samples, 1))
	return [X, labels], y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples, n_classes=10):
	# generate points in the latent space
	x_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	z_input = x_input.reshape(n_samples, latent_dim)
	# generate labels
	labels = randint(0, n_classes, n_samples)
	return [z_input, labels]

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(generator, latent_dim, n_samples):
	# generate points in latent space
	z_input, labels_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	images = generator.predict([z_input, labels_input])
	# create class labels
	y = zeros((n_samples, 1))
	return [images, labels_input], y

# train the generator and discriminator
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=100, n_batch=128):
	bat_per_epo = int(dataset[0].shape[0] / n_batch)
	half_batch = int(n_batch / 2)
	# manually enumerate epochs
	for i in range(n_epochs):
		# enumerate batches over the training set
		for j in range(bat_per_epo):
			# get randomly selected 'real' samples
			[X_real, labels_real], y_real = generate_real_samples(dataset, half_batch)
			# update discriminator model weights
			d_loss1, _ = d_model.train_on_batch([X_real, labels_real], y_real)
			# generate 'fake' examples
			[X_fake, labels], y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
			# update discriminator model weights
			d_loss2, _ = d_model.train_on_batch([X_fake, labels], y_fake)
			# prepare points in latent space as input for the generator
			[z_input, labels_input] = generate_latent_points(latent_dim, n_batch)
			# create inverted labels for the fake samples
			y_gan = ones((n_batch, 1))
			# update the generator via the discriminator's error
			g_loss = gan_model.train_on_batch([z_input, labels_input], y_gan)
			# summarize loss on this batch
			print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' %
				(i+1, j+1, bat_per_epo, d_loss1, d_loss2, g_loss))
	# save the generator model
	g_model.save('cgan_generator.h5')

# size of the latent space
latent_dim = 100
# create the discriminator
d_model = define_discriminator()
# create the generator
g_model = define_generator(latent_dim)
# create the gan
gan_model = define_gan(g_model, d_model)
# load image data
dataset = load_real_samples()
# train model
train(g_model, d_model, gan_model, dataset, latent_dim)

---

➡️ **Next / 下一步**: File 6 of 6